### 1. СОЗДАНИЕ ТАБЛИЦЫ И БАЗОВАЯ ЗАГРУЗКА

In [0]:
catalog_name = "task_1"
schema_name = "data_ingesting"
volume_name = "db_study"

table_name = f"{catalog_name}.{schema_name}.employees"

spark.sql(f"DROP TABLE IF EXISTS {table_name}")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

import pandas as pd

db = [
    ["1", "Kate", "HR", "1000"],
    ["2", "John", "IT", "2000"],
    ["3", "Mary", "IT", "2500"],
    ["4", "Mike", "HR", "1500"],
    ["5", "Jane", "Lawyer", "3000"]
]

columns = ["ID", "Name", "Department", "Salary"]
df = pd.DataFrame(db, columns=columns)
file_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/employees.csv"
df.to_csv(file_path, index=False)
display(spark.read.csv(file_path))



In [0]:

df_spark = spark.read.csv(file_path, header=True, inferSchema=True)
df_spark.write.format("delta").mode('overwrite').saveAsTable(table_name)


In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

### 2. ИМИТАЦИЯ DML

In [0]:
spark.sql(f""" UPDATE {catalog_name}.{schema_name}.employees
               SET Salary = Salary + 100
               WHERE Name = "Kate"
""")

display(spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.employees"))


In [0]:

spark.sql(f"""
          DELETE FROM {catalog_name}.{schema_name}.employees
          WHERE Name = "Mary"
          """)

display(spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.employees"))


In [0]:
spark.sql(f"""
          INSERT INTO {catalog_name}.{schema_name}.employees
          VALUES ("6", "Bob", "IT", "2000")
          """)

display(spark.sql(f"SELECT * FROM {catalog_name}.{schema_name}.employees"))

### 3. АУДИТ ИЗМЕНЕНИЙ (DESCRIBE)

In [0]:
%sql
DESCRIBE HISTORY employees

### 4. TIME TRAVEL AND RESTORE

In [0]:
%sql
SELECT *
FROM employees VERSION AS OF 0

**- Проверяем и доступность по нужному нам timestamp**

In [0]:
%sql
SELECT *
FROM employees
TIMESTAMP AS OF '2026-03-21T06:35:05.000+00:00'

In [0]:
%sql
RESTORE TABLE employees
TO VERSION AS OF 0;

SELECT *
FROM employees

### 5. SCHEMA EVOLUTION

In [0]:
new_data = [(7, "Bob", "IT", 3200, 500)]
new_columns = ["ID", "Name", "Department", "Salary", "Bonus"]

new_df = spark.createDataFrame(new_data, new_columns)



In [0]:
new_df.write.format("delta").mode("append").saveAsTable(table_name)

**- При попытке добавления данных упали с ошибкой из-за разных типов данных, 
поэтому создаем новый new_df с коректными типами данных для ID и добавляем ее к нашей таблице**

In [0]:
from pyspark.sql.functions import col


new_df_fixed = (
    new_df
    .withColumn("ID", col("ID").cast("int"))
    .withColumn("Name", col("Name").cast("string"))
    .withColumn("Department", col("Department").cast("string"))
    .withColumn("Salary", col("Salary").cast("int"))
    .withColumn("Bonus", col("Bonus").cast("int"))
)

In [0]:
new_df_fixed.write.format("delta").option("mergeSchema", "True").mode("append").saveAsTable(table_name)

In [0]:
%sql
SELECT *
FROM employees

### 6. TEMP VIEW и СЛИЯНИЕ ДАННЫХ

In [0]:
new_day = [
    (7,"Bob", "IT", 3200, 500),
    (8,"Mary", "HR", 2500, 300),
    (9,"Kate", "IT", 3000, 400),
    (10,"John", "IT", 3500, 600),
    (3,"Mary", "IT", 2500, 300)
    ]

columns = ["ID", "Name", "Department", "Salary", "Bonus"]

new_day_df = spark.createDataFrame(new_day, columns)


In [0]:
new_day_df.createOrReplaceTempView("new_day")


In [0]:
%sql
SELECT * FROM new_day;

In [0]:
%sql
MERGE INTO employees AS target
USING new_day AS source
ON target.id = source.id 

WHEN MATCHED THEN
UPDATE SET 
  target.Name = source.name
  ,target.Department = source.department
   ,target.Salary = source.salary
   ,target.Bonus = source.bonus

WHEN NOT MATCHED
THEN INSERT (ID, Name, Department, Salary, Bonus)
VALUES (source.ID, source.name, source.department, source.salary, source.bonus);
    

    


**- Проверяем на корректность обновлений**

In [0]:
%sql
SELECT *
FROM employees
ORDER BY id

### 7. ОПТИМИЗАЦИЯ 

In [0]:
%sql
ALTER TABLE employees
CLUSTER BY (Department, ID)

In [0]:
%sql

INSERT INTO employees VALUES (11, 'Sam', 'HR', 1700, NULL);
INSERT INTO employees VALUES (12, 'Leo', 'Finance', 3900, 300);
INSERT INTO employees VALUES (13, 'Nina', 'IT', 2600, 150);
INSERT INTO employees VALUES (14, 'Oleg', 'Lawyer', 3200, NULL);
INSERT INTO employees VALUES (15, 'Ira', 'HR', 1900, 120);
INSERT INTO employees VALUES (16, 'Greg', 'Finance', 1900, 120);

In [0]:
%sql
SELECT *
FROM employees

In [0]:
%sql
OPTIMIZE employees 

In [0]:
%sql
DESCRIBE HISTORY employees 

**Удалились 7 файлов**


![](/Volumes/task_1/data_ingesting/db_study/1.png)

**Полный вывод данной ячейки**

{"numRemovedFiles":"7","numRemovedBytes":"9786","p25FileSize":"2607","numDeletionVectorsRemoved":"0","conflictDetectionTimeMs":"46","minFileSize":"2607","p75FileSize":"2607","p50FileSize":"2607","numAddedBytes":"2607","numAddedFiles":"1","maxFileSize":"2607"}